# 04 — Baseline estese: finestre, β a tratti, β lineare nella mobilità

Obiettivo: chiudere i punti aperti prima del GP.
1. **Scelta della finestra**: quanto varia la mobilità, e quanto si confonde con il tempo t.
2. **Le due baseline mancanti** del documento di progetto: β costante a tratti (un valore per periodo fra due provvedimenti) e β lineare nella mobilità, β(t) = β₀ · (1 + Σ cₖ xₖ(t)).
3. **Sensibilità su σ** (latenza 4 gg, incubazione 5,2 gg di Li et al., 6,4 gg delle meta-analisi).

Impostazione identica alla sezione 6 del notebook 02, il riferimento per il GP: γ(t) e μ(t) dai dati, σ = 1/5.2, fitness normalizzata, training sui primi 2/3 della finestra, dati `clean`. Codice in `src/seird_gp/beta_models.py`, provvedimenti in `data/restrizioni_trento.csv`.

**Regola sul test**, uguale per tutte le baseline e per il GP: nessun dato futuro degli esiti (positivi, decessi, guariti); gli input esterni (mobilità osservata, date dei provvedimenti) si usano anche nel test. Un periodo della baseline a tratti che inizia dopo la fine del training non ha dati per stimare il suo β: resta al β dell'ultimo periodo del training.

I fit (35, circa 2 ore con 2 core) si lanciano con `scripts/run_baselines.py`; qui si leggono i risultati da `results/04_baseline_all.pkl`.

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from seird_gp.data import PROCESSED_DIR, RESULTS_DIR, load_mobility
from seird_gp.inputs import build_gp_inputs
from seird_gp.beta_models import load_restrictions

data = pd.read_csv(PROCESSED_DIR / "trento_clean.csv", index_col=0, parse_dates=True)
inputs = build_gp_inputs(load_mobility())
restr = load_restrictions()
WINDOWS = {"2020 (1/3–1/4)": ("2020-03-01", "2020-04-01"),
           "2021 (1/3–1/6)": ("2021-03-01", "2021-06-01"),
           "2022 (1/1–1/3)": ("2022-01-01", "2022-03-01"),
           "set20–gen21": ("2020-09-01", "2021-01-31"),
           "set20–feb21": ("2020-09-01", "2021-02-28")}
restr

## 1. Quanto varia la mobilità in ogni finestra

Per ogni finestra e per i due indici scelti nel notebook 03 (`lag0`):
- **escursione**: massimo − minimo, in punti percentuali. Se la mobilità non varia, il GP non può imparare niente su di essa;
- **R² con t**: quanto la mobilità è spiegata da una retta nel tempo. Se è alto, una formula in t imita quella in m(t) e il GP non può distinguerle (con t fra gli ingredienti);
- **|r| retail–work**: se è alto, i due indici sono intercambiabili (domanda D6 su quale conta).

In [ ]:
rows = []
for name, (a, b) in WINDOWS.items():
    x = inputs.loc[a:b]
    t = np.arange(len(x))
    row = {"finestra": name, "giorni": len(x)}
    for col in ["retail_lag0", "work_lag0"]:
        short = col.split("_")[0]
        row[f"{short} escursione"] = round(100 * (x[col].max() - x[col].min()))
        row[f"{short} R² con t"] = round(np.corrcoef(t, x[col])[0, 1] ** 2, 2)
    row["|r| retail–work"] = round(abs(x["retail_lag0"].corr(x["work_lag0"])), 2)
    rows.append(row)
mob = pd.DataFrame(rows).set_index("finestra")
mob

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.5))
a, b = "2020-08-15", "2021-03-15"
for col, c in [("retail_lag0", "#1f4e79"), ("work_lag0", "#c47f2c")]:
    ax.plot(100 * inputs.loc[a:b, col], color=c, label=col.split("_")[0])
for _, r in restr[(restr.inizio >= a) & (restr.inizio <= b)].iterrows():
    ax.axvline(r.inizio, color="#a33", lw=0.6, ls="--")
    ax.text(r.inizio, 2, r.livello, rotation=90, fontsize=7, va="bottom", color="#a33")
ax.axvspan("2020-09-01", "2021-01-31", color="#eee", zorder=0, label="set20–gen21")
ax.set_ylabel("Variazione % rispetto a gen–feb 2020")
ax.legend(frameon=False, fontsize=8)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig(RESULTS_DIR / "04_mobilita_seconda_ondata.png", dpi=150)

## 2. Confronto fra le baseline

In [ ]:
res = pickle.load(open(RESULTS_DIR / "04_baseline_all.pkl", "rb"))
REF = "σ = 1/5.2"
rows = []
for (w, m, s), r in res.items():
    if s != REF:
        continue
    rows.append({"finestra": w, "baseline": m, "n. parametri": r.extra["n_params"],
                 "fitness_norm train": r.rmsd.loc["train", "fitness_norm"],
                 "fitness_norm test": r.rmsd.loc["test", "fitness_norm"],
                 "RMSD D test": r.rmsd.loc["test", "D"], "RMSD I test": r.rmsd.loc["test", "I"],
                 "al bordo": ", ".join(r.extra["at_bound"])})
order = {"costante": 0, "a tratti": 1, "lineare retail": 2, "lineare work": 3, "lineare retail+work": 4}
comp = (pd.DataFrame(rows).assign(o=lambda d: d.baseline.map(order))
        .sort_values(["finestra", "o"]).drop(columns="o").set_index(["finestra", "baseline"]))
comp.to_csv(RESULTS_DIR / "04_baseline_estese.csv")
comp.round(3)

In [ ]:
# parametri stimati della forma di β (coefficienti della mobilità e β dei periodi)
pars = {}
for (w, m, s), r in res.items():
    if s == REF and m != "costante":
        pars[(w, m)] = {k: v for k, v in r.params.items() if k.startswith(("beta_", "beta0", "c_"))}
pd.DataFrame(pars).T.round(3)

In [ ]:
W = "set20–gen21"
a, b = WINDOWS[W]
obs = data.loc[a:b]
colors = {"costante": "#999", "a tratti": "#a33", "lineare retail": "#1f4e79",
          "lineare work": "#c47f2c", "lineare retail+work": "#3a8a5c"}
fig, ax = plt.subplots(3, 1, figsize=(11, 10), sharex=True)
ax[0].plot(obs.index, obs.totale_positivi, "o", ms=2.5, color="#bbb", label="dati")
ax[1].plot(obs.index, obs.deceduti, "o", ms=2.5, color="#bbb", label="dati")
for m, c in colors.items():
    r = res[(W, m, REF)]
    ax[0].plot(r.simulation["I"], color=c, label=m)
    ax[1].plot(r.simulation["D"], color=c)
    ax[2].plot(r.extra["beta"], color=c, label=m)
for x in ax:
    x.axvspan(res[(W, "costante", REF)].train_end, obs.index[-1], color="#f4e3c1", alpha=0.6)
    for d in restr.inizio[(restr.inizio > a) & (restr.inizio <= b)]:
        x.axvline(d, color="#a33", lw=0.5, ls="--")
    x.spines[["top", "right"]].set_visible(False)
ax[0].set_ylabel("Positivi attuali"); ax[1].set_ylabel("Decessi (cumulativi)"); ax[2].set_ylabel("β(t)")
ax[0].legend(frameon=False, fontsize=8, ncol=3)
fig.tight_layout(); fig.savefig(RESULTS_DIR / "04_baseline_seconda_ondata.png", dpi=150)

## 3. Sensibilità su σ (β costante)

In [ ]:
rows = []
for (w, m, s), r in res.items():
    if m == "costante":
        rows.append({"finestra": w, "σ": s, "beta": r.params["beta"], "E0": r.params["E0"],
                     "fitness_norm train": r.rmsd.loc["train", "fitness_norm"],
                     "fitness_norm test": r.rmsd.loc["test", "fitness_norm"]})
sens = pd.DataFrame(rows).sort_values(["finestra", "σ"]).set_index(["finestra", "σ"])
sens.round(3)